# 5a. Long Term Memory (continued)

Continuing from the previous notebook, build a memory agent to incorporate a profile and collection.

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 16/06/2026  | Martin | CREATE   | Started mini project: memory agent for a to-do list | 
| 07/07/2026  | Martin | UPDATE   | Built schemas and explore trustcall memory viewing | 

# Content

* [Introduction](#introduction)
* [Visibility into Trustcall](#visibility-into-trustcall)
* [Create the Agent](#create-the-agent)

# Introduction

The agent will have long-term memory - `task_demon`

Additional functionality compared to previous agents

- Previous chatbots had to have their memories passed in before recording. `task_demon` will decide **when** to save memories
- It will also decide to save either as a user profile or a collection

In other words, it will have semantic memory (how to perform tasks) and procedural memory (when to perform tasks)

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

---

# Visibility into Trustcall

Want to view exactly what changes trustcall made to the memory.

In [2]:
from pydantic import BaseModel, Field, SecretStr

class Memory(BaseModel):
    """Main memory content of the user"""
    content: str = Field(
        description=(
            "The main content of the memory. For example: User expressed "
            "interest in learning Japanese"
        )
    )

class MemoryCollection(BaseModel):
    """A colleection of memories stored for the user"""
    memories: list[Memory] = Field(description="A list of memories about the user")

Create a *listener* that will pass runs from the extractor to a class `Spy` that will extract information about what tool calls were made by Trustcall. 

In [3]:
from trustcall import create_extractor
from langchain_groq import ChatGroq

class Spy:
    def __init__(self):
        self.called_tools = []

    def __call__(self, run):
        """Collect infromation about the tool calls made by the extractor"""
        q = [run]
        while q:
            r = q.pop()
            if r.child_runs:
                q.extend(r.child_runs)
            if r.run_type == "chat_model":
                self.called_tools.append(
                    r.outputs['generations'][0][0]['message']['kwargs']['tool_calls']
                )

# Define the listener
spy = Spy()

# Define our model
API_KEY = SecretStr(os.getenv("GROQ_API_KEY", ""))
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=API_KEY
)

# Define the extractor
trustcall_extractor = create_extractor(
    model,
    tools=[Memory],
    tool_choice="Memory",
    enable_inserts=True
)

# Add the spy as a listener
trustcall_extractor_see_all_tool_calls = trustcall_extractor.with_listeners(on_end=spy)

Example of trustcall extracting memories from conversations

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

instruction = """Extract memories from the following conversation:"""

conversation = [
    HumanMessage("Hi I'm Martin"),
    AIMessage("Hi Martin, nice to meet you"),
    HumanMessage("This morning I went to each breakfast at Chin Mee Chin")
]

result = trustcall_extractor.invoke({
    "messages": [SystemMessage(content=instruction)] + conversation
})

for m in result['messages']:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (1gma7qazf)
 Call ID: 1gma7qazf
  Args:
    content: User went to breakfast at Chin Mee Chin this morning


In [5]:
# Metadata contained in the tool call
for m in result['response_metadata']:
    print(m)

{'id': '1gma7qazf'}


In [6]:
# Update the conversation
updated_conversation = [
    AIMessage("That's great what did you do after"),
    HumanMessage("I took my dog Katie out for a walk"),
    AIMessage("That's great! What else is on your mind"),
    HumanMessage("I'm thinking about what to get my sister for her birthday"),
]

# Update the instructions
sys_msg = """Update existing memories and create new ones based on the follwing conversation"""

# Save existing memroies giving them an ID, key (tool_name), and value
tool_name = "Memory"
existing_memories = [(str(i), tool_name, memory.model_dump()) for i, memory in enumerate(result["responses"])] if result["responses"] else None
existing_memories # NOTE: Still the same memory here

[('0',
  'Memory',
  {'content': 'User went to breakfast at Chin Mee Chin this morning'})]

In [7]:
result = trustcall_extractor_see_all_tool_calls.invoke({
    "messages": updated_conversation,
    "existing": existing_memories
})

In [8]:
for m in result['response_metadata']:
    print(m)

{'id': 'jsv22s7v5'}
{'id': 'etcn32ysz', 'json_doc_id': '0'}


The `'json_doc_id'` variable states that the memory with id '0' was updated

In [9]:
# Messages contain the tool calls
for m in result["messages"]:
    m.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Memory (jsv22s7v5)
 Call ID: jsv22s7v5
  Args:
    content: User is thinking about what to get their sister for her birthday
  Memory (etcn32ysz)
 Call ID: etcn32ysz
  Args:
    content: User went to breakfast at Chin Mee Chin this morning
    -: User is thinking about what to get their sister for her birthday


In [10]:
# Parsed responses
for m in result["responses"]:
    print(m)

content='User is thinking about what to get their sister for her birthday'
content='User went to breakfast at Chin Mee Chin this morning'


In [11]:
# Inspect the tool calls made by Trustcall
spy.called_tools

[[{'name': 'Memory',
   'args': {'content': 'User is thinking about what to get their sister for her birthday'},
   'id': 'jsv22s7v5',
   'type': 'tool_call'},
  {'name': 'PatchDoc',
   'args': {'json_doc_id': '0',
    'patches': [{'op': 'add',
      'path': '/-',
      'value': 'User is thinking about what to get their sister for her birthday'}],
    'planned_edits': 'add a new memory content'},
   'id': 'etcn32ysz',
   'type': 'tool_call'}]]

In [13]:
def extract_tool_info(tool_calls, schema_name="Memory"):

    # Initialise a list of changes
    changes = []

    for call_group in tool_calls:
        for call in call_group:
            if call['name'] == "PatchDoc":
                changes.append({
                    "type": "update",
                    'doc_id': call['args']['json_doc_id'],
                    'planned_edits': call['args']['planned_edits'],
                    'value': call['args']['patches'][0]['value']
                })
            elif call['name'] == schema_name:
                changes.append({
                    'type': 'new',
                    'value': call['args']
                })

    # Format results as a single string
    result_parts = []
    for change in changes:
        if change['type'] == 'update':
            result_parts.append(
                f"Document {change['doc_id']} updated:\n"
                f"Plan: {change['planned_edits']}\n"
                f"Added content: {change['value']}"
            )
        else:
            result_parts.append(
                f"New {schema_name} created:\n"
                f"Content: {change['value']}"
            )

    return "\n\n".join(result_parts)

# Inspect spy.called_tools to see exactly what happened during the extraction
schema_name = "Memory"
changes = extract_tool_info(spy.called_tools, schema_name)
print(changes)

New Memory created:
Content: {'content': 'User is thinking about what to get their sister for her birthday'}

Document 0 updated:
Plan: add a new memory content
Added content: User is thinking about what to get their sister for her birthday


# Create the Agent

Implement the agents with ReAct framework. The agent can perform 3 different set of "actions" (other agents)

1. Update the users `profile` with new information
2. Update an item is the users `collection`
3. Update the set of `instructions` on how to update items in the ToDo list

In [ ]:
from typing import TypedDict, Literal

class UpdateMemory(TypedDict):
    """Decision on what memory type to be updated"""
    update_type: Literal['user', 'todo', 'instructions']

## Graph definition

In [ ]:
from typing import Optional
from pydantic import SecretStr, BaseModel
from langchain_groq import ChatGroq
from datetime import datetime

API_KEY = SecretStr(os.getenv("GROQ_API_KEY", ""))
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=API_KEY
)

# User profile schema
class Profile(BaseModel):
    """This is the profile of the user you are chatting with"""
    name: Optional[str] = Field(description="The user's name", default=None)
    location: Optional[str] = Field(description="The user's location", default=None)
    job: Optional[str] = Field(description="The user's job", default=None)
    connections: list[str] = Field(
        description="Personal connection of the user, such as family members, friends, or coworkers",
        default_factory=list
    )
    interests: list[str] = Field(
        description="Interests that the user has", 
        default_factory=list
    )

# ToDo Schema
class ToDo(BaseModel):
    task: str = Field(description="The task to be completed.")
    time_to_complete: Optional[int] = Field(description="Estimated time to complete the task (minutes).")
    deadline: Optional[datetime] = Field(
        description="When the task needs to be completed by (if applicable)",
        default=None
    )
    solutions: list[str] = Field(
        description="List of specific, actionable solutions (e.g., specific ideas, service providers, or concrete options relevant to completing the task)",
        min_length=1,
        default_factory=list
    )
    status: Literal["not started", "in progress", "done", "archived"] = Field(
        description="Current status of the task",
        default="not started"
    )

# Create the Trustcall extractor
profile_extractor = create_extractor(
    model,
    tools=[Profile],
    tool_choice="Profile",
)

In [ ]:
%load_ext watermark
%watermark